In [ ]:
#!/usr/bin/env python3
"""
gsMap Mouse Whole-Brain MERFISH Mapping Pipeline
==================================================
GWAS + mouse MERFISH whole-brain ST data -> cross-species spatial mapping

Steps:
  0. Prepare mouse MERFISH data (downsample, QC, format)
  1-5. gsMap core pipeline (latent representations, gene mapping,
       LD score generation, spatial LDSC, Cauchy combination)
  6. Visualization (3-panel brain map, cell class bar chart,
     significant cells map)
"""

import os
import sys
import time
import subprocess
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path


# ========================= Configuration =========================

# --- Required paths (fill in before running) ---
GWAS_SUMSTATS    = "./data/gwas/your_gwas_summary_data.sumstats.gz"   # Pre-formatted GWAS summary statistics (.sumstats.gz)
TRAIT_NAME       = "demo_mouse"   # Trait identifier (e.g. "T2DBrain_mice")
GSMAP_RESOURCE   = "./data/gsmap_resource/"   # gsMap resource directory
PROJECT_DIR      = "./results/mouse/"   # Output project directory

# --- Mouse MERFISH data ---
MOUSE_MERFISH_H5AD = "./data/visium/your_mouse_data.h5ad"   # Mouse brain MERFISH expression h5ad
MOUSE_METADATA_CSV = "./data/visium/your_mouse_data.csv"   # Cell metadata CSV with class, subclass, x, y columns

# --- Downsampling ---
MAX_CELLS_PER_SUBCLASS = 3000

# --- Compute resources ---
NUM_PARALLEL_CHROMS = 22
NUM_PROCESSES_LDSC  = 32


# ========================= Auto-configured =========================

GSMAP_BIN = os.path.join(os.path.dirname(sys.executable), "gsmap")
if not os.path.exists(GSMAP_BIN):
    GSMAP_BIN = "gsmap"


# ========================= Utilities =========================

def log(msg, level="INFO"):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {level} | {msg}")


def run_gsmap(args, step_name):
    """Run a gsmap CLI command and return success status."""
    log(f"Starting {step_name}...")
    cmd = [GSMAP_BIN] + args
    process = subprocess.run(cmd, capture_output=True, text=True)
    if process.returncode == 0:
        log(f"{step_name} completed")
        return True
    else:
        log(f"{step_name} failed", "ERROR")
        log(process.stderr[-1000:], "ERROR")
        return False


def _run_chrom_global(args):
    """Run LD score generation for a single chromosome (for parallel dispatch)."""
    gsmap_bin, workdir, sample_name, chrom, resource = args
    cmd = [gsmap_bin, "run_generate_ldscore",
           "--workdir", workdir,
           "--sample_name", sample_name,
           "--chrom", str(chrom),
           "--bfile_root", f"{resource}/LD_Reference_Panel/1000G_EUR_Phase3_plink/1000G.EUR.QC",
           "--keep_snp_root", f"{resource}/LDSC_resource/hapmap3_snps/hm",
           "--gtf_annotation_file", f"{resource}/genome_annotation/gtf/gencode.v46lift37.basic.annotation.gtf",
           "--gene_window_size", "50000"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return chrom, result.returncode


# ========================= Step 0: Prepare Mouse Data =========================

def prepare_mouse_data():
    """Read mouse MERFISH data, downsample, add spatial coords and annotations."""
    log("=" * 60)
    log("Step 0: Prepare mouse MERFISH data")
    log("=" * 60)

    output_path = os.path.join(PROJECT_DIR, f"mouse_brain_for_gsmap_{TRAIT_NAME}.h5ad")
    if os.path.exists(output_path):
        log(f"Data already exists, skipping: {output_path}")
        return output_path

    # Read metadata
    log(f"Reading metadata: {MOUSE_METADATA_CSV}")
    meta = pd.read_csv(MOUSE_METADATA_CSV)
    log(f"Metadata rows: {len(meta)}")

    # QC: remove unannotated cells
    meta_clean = meta.dropna(subset=["class", "subclass", "x", "y"])
    log(f"Annotated cells: {len(meta_clean)}")

    # Downsample per subclass
    log(f"Downsampling: max {MAX_CELLS_PER_SUBCLASS} cells per subclass")
    np.random.seed(42)
    sample_indices = []
    for sc_type in meta_clean["subclass"].unique():
        idx = meta_clean[meta_clean["subclass"] == sc_type].index.tolist()
        n_sample = min(len(idx), MAX_CELLS_PER_SUBCLASS)
        sampled = np.random.choice(idx, n_sample, replace=False)
        sample_indices.extend(sampled)

    meta_sub = meta_clean.loc[sample_indices].copy()
    log(f"Cells after downsampling: {len(meta_sub)}")
    log(f"Subclass count: {meta_sub['subclass'].nunique()}")

    # Read expression data
    log(f"Reading MERFISH expression: {MOUSE_MERFISH_H5AD}")
    adata_full = sc.read_h5ad(MOUSE_MERFISH_H5AD, backed="r")

    # Match cells
    common_cells = adata_full.obs_names.intersection(meta_sub["cell_label"].astype(str))
    log(f"Matched cells: {len(common_cells)}")

    if len(common_cells) > 0:
        meta_sub_matched = meta_sub.set_index("cell_label")
        meta_sub_matched.index = meta_sub_matched.index.astype(str)
        meta_sub_matched = meta_sub_matched.loc[common_cells]
        adata_sub = adata_full[common_cells, :].to_memory()
    else:
        log("Falling back to index-based matching...")
        meta_sub_sorted = meta_sub.sort_index()
        valid_idx = [i for i in meta_sub_sorted.index if i < adata_full.n_obs]
        log(f"Valid indices: {len(valid_idx)}")
        adata_sub = adata_full[valid_idx, :].to_memory()
        meta_sub_matched = meta_sub_sorted.loc[valid_idx]

    # Add spatial coordinates and annotations
    log("Setting spatial coordinates and annotations...")
    adata_sub.obsm["spatial"] = meta_sub_matched[["x", "y"]].values.astype(float)
    adata_sub.obs["annotation"] = meta_sub_matched["subclass"].values
    adata_sub.obs["class"] = meta_sub_matched["class"].values
    adata_sub.obs["brain_region"] = meta_sub_matched["supertype"].values

    # Convert gene names
    log("Converting gene names...")
    adata_sub.var_names = adata_sub.var["gene_symbol"].values.astype(str)
    adata_sub.var_names_make_unique()
    log(f"Gene name examples: {adata_sub.var_names[:5].tolist()}")

    # Store count layer
    adata_sub.layers["count"] = adata_sub.X.copy()

    # Save
    adata_sub.write_h5ad(output_path)
    log(f"Mouse data prepared: {output_path}")
    log(f"  Cells: {adata_sub.n_obs}, Genes: {adata_sub.n_vars}")
    log(f"  Classes: {adata_sub.obs['class'].nunique()}")

    return output_path


# ========================= Steps 1-5: gsMap Core Pipeline =========================

def run_gsmap_pipeline(h5ad_path):
    """Run gsMap steps 1-5 with checkpoint resumption."""
    sample_name = f"mouse_brain_{TRAIT_NAME}"
    workdir = os.path.join(PROJECT_DIR, "results")
    homolog_file = os.path.join(GSMAP_RESOURCE, "homologs/mouse_human_homologs.txt")

    os.makedirs(workdir, exist_ok=True)

    # --- Step 1: Find latent representations ---
    step1_check = os.path.join(workdir, sample_name, "find_latent_representations")
    if os.path.exists(step1_check):
        log("Step 1 already completed, skipping")
    else:
        try:
            adata_check = sc.read_h5ad(h5ad_path, backed="r")
            if "latent_GVAE" in adata_check.obsm:
                log("Step 1 already completed (latent_GVAE found), skipping")
            else:
                raise Exception()
        except Exception:
            log("=" * 60)
            if not run_gsmap(["run_find_latent_representations",
                              "--workdir", workdir,
                              "--sample_name", sample_name,
                              "--input_hdf5_path", h5ad_path,
                              "--annotation", "annotation",
                              "--data_layer", "count"],
                             "Step 1: find_latent_representations"):
                return False, sample_name, workdir

    # --- Step 2: Latent to gene ---
    latent_to_gene_dir = os.path.join(workdir, sample_name, "latent_to_gene")
    if os.path.exists(latent_to_gene_dir) and len(os.listdir(latent_to_gene_dir)) > 0:
        log("Step 2 already completed, skipping")
    else:
        log("=" * 60)
        if not run_gsmap(["run_latent_to_gene",
                          "--workdir", workdir,
                          "--sample_name", sample_name,
                          "--annotation", "annotation",
                          "--latent_representation", "latent_GVAE",
                          "--num_neighbour", "51",
                          "--num_neighbour_spatial", "201",
                          "--homolog_file", homolog_file],
                         "Step 2: latent_to_gene"):
            return False, sample_name, workdir

    # --- Step 3: Generate LD scores (22 chromosomes in parallel) ---
    log("=" * 60)
    log("Step 3: generate_ldscore (22 chromosomes in parallel)")

    ldscore_dir = os.path.join(workdir, sample_name, "generate_ldscore")
    step3_done = False
    if os.path.exists(ldscore_dir):
        chunks = [f for f in os.listdir(ldscore_dir) if f.startswith(f"{sample_name}_chunk")]
        if chunks:
            sample_chunk = os.path.join(ldscore_dir, chunks[0])
            if os.path.exists(sample_chunk) and len(os.listdir(sample_chunk)) >= 66:
                step3_done = True
                log("Step 3 already completed, skipping")

    if not step3_done:
        from concurrent.futures import ProcessPoolExecutor, as_completed

        chrom_args = [(GSMAP_BIN, workdir, sample_name, c, GSMAP_RESOURCE) for c in range(1, 23)]
        with ProcessPoolExecutor(max_workers=NUM_PARALLEL_CHROMS) as executor:
            futures = {executor.submit(_run_chrom_global, args): args[3] for args in chrom_args}
            for future in as_completed(futures):
                chrom, rc = future.result()
                status = "OK" if rc == 0 else "FAIL"
                log(f"  {status} chr{chrom}")

        # Verify completeness and rerun missing chromosomes
        if os.path.exists(ldscore_dir):
            chunks = [f for f in os.listdir(ldscore_dir) if f.startswith(f"{sample_name}_chunk")]
            if chunks:
                sample_chunk = os.path.join(ldscore_dir, chunks[0])
                if os.path.exists(sample_chunk):
                    n_files = len(os.listdir(sample_chunk))
                    if n_files < 66:
                        log(f"Incomplete ({n_files}/66), rerunning missing...", "WARN")
                        chroms_found = set()
                        for f in os.listdir(sample_chunk):
                            if ".l2.ldscore.feather" in f:
                                chroms_found.add(int(f.split(".")[1]))
                        missing = set(range(1, 23)) - chroms_found
                        if missing:
                            missing_args = [(GSMAP_BIN, workdir, sample_name, c, GSMAP_RESOURCE) for c in missing]
                            with ProcessPoolExecutor(max_workers=len(missing)) as executor:
                                futures = {executor.submit(_run_chrom_global, args): args[3] for args in missing_args}
                                for future in as_completed(futures):
                                    chrom, rc = future.result()
                                    status = "OK" if rc == 0 else "FAIL"
                                    log(f"  Rerun {status} chr{chrom}")

        log("Step 3 completed")

    # --- Step 4: Spatial LDSC ---
    ldsc_result = os.path.join(workdir, sample_name, "spatial_ldsc",
                               f"{sample_name}_{TRAIT_NAME}.csv.gz")
    if os.path.exists(ldsc_result):
        log("Step 4 already completed, skipping")
    else:
        log("=" * 60)
        if not run_gsmap(["run_spatial_ldsc",
                          "--workdir", workdir,
                          "--sample_name", sample_name,
                          "--trait_name", TRAIT_NAME,
                          "--sumstats_file", GWAS_SUMSTATS,
                          "--w_file", f"{GSMAP_RESOURCE}/LDSC_resource/weights_hm3_no_hla/weights.",
                          "--num_processes", str(NUM_PROCESSES_LDSC)],
                         "Step 4: spatial_ldsc"):
            return False, sample_name, workdir

    # --- Step 5: Cauchy combination ---
    cauchy_result = os.path.join(workdir, sample_name, "cauchy_combination",
                                 f"{sample_name}_{TRAIT_NAME}.Cauchy.csv.gz")
    if os.path.exists(cauchy_result):
        log("Step 5 already completed, skipping")
    else:
        log("=" * 60)
        if not run_gsmap(["run_cauchy_combination",
                          "--workdir", workdir,
                          "--sample_name", sample_name,
                          "--trait_name", TRAIT_NAME,
                          "--annotation", "annotation"],
                         "Step 5: cauchy_combination"):
            return False, sample_name, workdir

    log("gsMap core pipeline completed")
    return True, sample_name, workdir


# ========================= Step 6: Visualization =========================

def run_visualization(h5ad_path, sample_name, workdir):
    """Generate publication-quality figures from gsMap results."""
    import matplotlib
    matplotlib.rcParams["font.family"] = "Arial"
    matplotlib.rcParams["pdf.fonttype"] = 42
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    log("=" * 60)
    log("Step 6: Visualization")
    log("=" * 60)

    fig_dir = os.path.join(PROJECT_DIR, "figures")
    os.makedirs(fig_dir, exist_ok=True)

    adata = sc.read_h5ad(h5ad_path)
    ldsc_path = os.path.join(workdir, sample_name, "spatial_ldsc",
                             f"{sample_name}_{TRAIT_NAME}.csv.gz")
    cauchy_path = os.path.join(workdir, sample_name, "cauchy_combination",
                               f"{sample_name}_{TRAIT_NAME}.Cauchy.csv.gz")

    spatial_ldsc = pd.read_csv(ldsc_path)
    cauchy = pd.read_csv(cauchy_path)

    # Align cells
    spatial_ldsc = spatial_ldsc.set_index("spot")
    common = adata.obs_names.intersection(spatial_ldsc.index)
    adata = adata[common, :]
    spatial_ldsc = spatial_ldsc.loc[common]

    adata.obs = adata.obs.copy()
    adata.obs["gsmap_p"] = spatial_ldsc["p"].values
    adata.obs["gsmap_neglog10p"] = -np.log10(spatial_ldsc["p"].values + 1e-300)

    coords = adata.obsm["spatial"]
    clip_val = np.percentile(adata.obs["gsmap_neglog10p"], 99)
    plot_vals = np.clip(adata.obs["gsmap_neglog10p"].values, 0, clip_val)

    # --- Figure 1: Whole-brain 3-panel ---
    fig, axes = plt.subplots(1, 3, figsize=(30, 9))

    # Panel 1: Cell classes
    ax = axes[0]
    ax.set_facecolor("white")
    ax.scatter(coords[:, 0], coords[:, 1], c="#E8E8E8", s=0.5, alpha=0.5, rasterized=True)
    color_map = {}
    classes = adata.obs["class"].cat.categories if hasattr(adata.obs["class"], "cat") else adata.obs["class"].unique()
    for c in classes:
        if "Glut" in str(c):
            color_map[c] = "#3498db"
        elif "GABA" in str(c):
            color_map[c] = "#e74c3c"
        elif any(x in str(c) for x in ["Astro", "OPC", "Oligo", "Vascular", "Immune", "OEC"]):
            color_map[c] = "#2ecc71"
        else:
            color_map[c] = "#95a5a6"
    cell_colors = [color_map.get(c, "#95a5a6") for c in adata.obs["class"]]
    ax.scatter(coords[:, 0], coords[:, 1], c=cell_colors, s=0.3, alpha=0.4, rasterized=True)
    legend_elements = [Patch(facecolor="#3498db", label="Glutamatergic"),
                       Patch(facecolor="#e74c3c", label="GABAergic"),
                       Patch(facecolor="#2ecc71", label="Non-neuronal"),
                       Patch(facecolor="#95a5a6", label="Other")]
    ax.legend(handles=legend_elements, fontsize=10, loc="lower right")
    ax.set_title("Tissue Section\nMouse Brain MERFISH", fontsize=14, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")

    # Panel 2: Spatial regions
    ax = axes[1]
    ax.set_facecolor("white")
    ax.scatter(coords[:, 0], coords[:, 1], c="#E8E8E8", s=0.5, alpha=0.5, rasterized=True)
    region_colors = {"Cortex": "#3498db", "Thalamus": "#e74c3c", "Hippocampus": "#2ecc71",
                     "Midbrain": "#f39c12", "Cerebellum": "#9b59b6", "Brainstem": "#1abc9c",
                     "Striatum": "#e67e22", "Other": "#95a5a6"}
    cell_region_colors = []
    for cls in adata.obs["class"]:
        cls_str = str(cls)
        if "IT-ET" in cls_str or "CTX" in cls_str or "NP-CT" in cls_str:
            cell_region_colors.append(region_colors["Cortex"])
        elif "TH" in cls_str:
            cell_region_colors.append(region_colors["Thalamus"])
        elif "DG" in cls_str or "IMN" in cls_str:
            cell_region_colors.append(region_colors["Hippocampus"])
        elif "MB" in cls_str:
            cell_region_colors.append(region_colors["Midbrain"])
        elif "CB" in cls_str:
            cell_region_colors.append(region_colors["Cerebellum"])
        elif "MY" in cls_str or "P " in cls_str or "HB" in cls_str:
            cell_region_colors.append(region_colors["Brainstem"])
        elif "CNU" in cls_str or "LSX" in cls_str:
            cell_region_colors.append(region_colors["Striatum"])
        else:
            cell_region_colors.append(region_colors["Other"])
    ax.scatter(coords[:, 0], coords[:, 1], c=cell_region_colors, s=0.3, alpha=0.5, rasterized=True)
    legend_regions = [Patch(facecolor=v, label=k) for k, v in region_colors.items()]
    ax.legend(handles=legend_regions, fontsize=9, loc="lower right")
    ax.set_title("Spatial Regions", fontsize=14, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")

    # Panel 3: gsMap results
    ax = axes[2]
    ax.set_facecolor("white")
    ax.scatter(coords[:, 0], coords[:, 1], c="#E8E8E8", s=0.5, alpha=0.5, rasterized=True)
    sc_plot = ax.scatter(coords[:, 0], coords[:, 1], c=plot_vals, cmap="Reds",
                         s=0.5, alpha=0.85, vmin=0, vmax=clip_val, rasterized=True)
    cbar = plt.colorbar(sc_plot, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label("-log10(P value)", fontsize=12)
    ax.set_title(f"gsMap: {TRAIT_NAME}\nSpatial Association", fontsize=14, fontweight="bold")
    ax.set_aspect("equal")
    ax.axis("off")

    plt.suptitle(f"Genetically Informed Spatial Mapping of {TRAIT_NAME} on Mouse Whole Brain",
                 fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, f"mouse_wholebrain_{TRAIT_NAME}.pdf"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(fig_dir, f"mouse_wholebrain_{TRAIT_NAME}.png"), dpi=300, bbox_inches="tight")
    plt.close()
    log("Figure 1: whole-brain 3-panel saved")

    # --- Figure 2: Cell class association bar chart ---
    class_results = []
    for cls in adata.obs["class"].unique():
        mask = (adata.obs["class"] == cls).values
        pvals = adata.obs.loc[mask, "gsmap_p"].values
        weights = np.ones(len(pvals)) / len(pvals)
        cauchy_stats = np.sum(weights * np.tan((0.5 - pvals) * np.pi))
        cauchy_p = 0.5 - np.arctan(cauchy_stats) / np.pi
        class_results.append({"Cell Class": cls, "p_cauchy": cauchy_p, "n_cells": mask.sum()})

    class_df = pd.DataFrame(class_results).sort_values("p_cauchy")
    class_df["neglog10p"] = -np.log10(class_df["p_cauchy"] + 1e-300)

    log(f"\nCell class association results:\n{class_df.to_string(index=False)}")

    fig, ax = plt.subplots(figsize=(10, 10))
    n_types = len(class_df)
    colors = ["#c0392b" if p < 0.05 / n_types else "#e74c3c" if p < 0.05 else "#bdc3c7"
              for p in class_df["p_cauchy"]]
    ax.barh(range(n_types), class_df["neglog10p"], color=colors, edgecolor="white")
    ax.set_yticks(range(n_types))
    ax.set_yticklabels(class_df["Cell Class"], fontsize=8)
    ax.set_xlabel("-log10(P value)", fontsize=12)
    ax.set_title(f"Mouse Brain Cell Class Association with {TRAIT_NAME}", fontsize=14, fontweight="bold")
    ax.axvline(x=-np.log10(0.05 / n_types), color="red", linestyle="--", linewidth=1, label="Bonferroni")
    ax.axvline(x=-np.log10(0.05), color="orange", linestyle="--", linewidth=1, label="Nominal")
    ax.legend(loc="lower right")
    ax.invert_yaxis()
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, f"mouse_cellclass_{TRAIT_NAME}.pdf"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(fig_dir, f"mouse_cellclass_{TRAIT_NAME}.png"), dpi=300, bbox_inches="tight")
    plt.close()
    log("Figure 2: cell class bar chart saved")

    # --- Figure 3: Significant cells brain map ---
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.set_facecolor("white")
    nonsig = adata.obs["gsmap_p"].values >= 0.01
    ax.scatter(coords[nonsig, 0], coords[nonsig, 1], c="#DDDDDD", s=0.3, alpha=0.3,
               rasterized=True, label=f"Not significant ({nonsig.sum():,})")
    sig = adata.obs["gsmap_p"].values < 0.01
    sig_vals = adata.obs.loc[sig, "gsmap_neglog10p"].values
    sc_sig = ax.scatter(coords[sig, 0], coords[sig, 1], c=sig_vals, cmap="Reds",
                        s=1.5, alpha=0.9, rasterized=True, vmin=2)
    cbar = plt.colorbar(sc_sig, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label("-log10(P value)", fontsize=12)
    ax.set_title(f"Significant Cells (P<0.01): {sig.sum():,} / {len(adata):,}",
                 fontsize=14, fontweight="bold")
    ax.legend(loc="lower right", fontsize=10, markerscale=10)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, f"mouse_significant_{TRAIT_NAME}.pdf"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(fig_dir, f"mouse_significant_{TRAIT_NAME}.png"), dpi=300, bbox_inches="tight")
    plt.close()
    log("Figure 3: significant cells map saved")

    # Save cell class results
    cauchy_sorted = cauchy.sort_values("p_cauchy")
    log(f"\nCell subclass top 20:\n{cauchy_sorted.head(20).to_string(index=False)}")

    class_df.to_csv(os.path.join(fig_dir, f"mouse_cellclass_results_{TRAIT_NAME}.csv"), index=False)

    log(f"\nAll figures saved to: {fig_dir}")


# ========================= Main =========================

def main():
    log("=" * 60)
    log("gsMap Mouse Whole-Brain MERFISH Pipeline")
    log(f"Trait: {TRAIT_NAME}")
    log(f"Project directory: {PROJECT_DIR}")
    log("=" * 60)

    os.makedirs(PROJECT_DIR, exist_ok=True)

    # Check GWAS file
    if not os.path.exists(GWAS_SUMSTATS):
        log(f"GWAS file not found: {GWAS_SUMSTATS}", "ERROR")
        return

    # Step 0: Prepare data
    h5ad_path = prepare_mouse_data()
    if not h5ad_path:
        log("Data preparation failed", "ERROR")
        return

    # Steps 1-5: gsMap core
    success, sample_name, workdir = run_gsmap_pipeline(h5ad_path)
    if not success:
        log("gsMap pipeline failed", "ERROR")
        return

    # Step 6: Visualization
    run_visualization(h5ad_path, sample_name, workdir)

    log("=" * 60)
    log("Pipeline completed")
    log("=" * 60)


if __name__ == "__main__":
    main()